# 4. Exploratory Data Analysis and Visualization
Run `03_data_wrangling.ipynb` first. The charts below are generated from the local processed dataset and saved into `outputs/figures/`.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("..")
DATA = ROOT / "data" / "processed" / "spacex_falcon9_launches.csv"
OUT = ROOT / "outputs" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA, parse_dates=["date_utc"])
labelled = df[df["landing_attempt"].astype(str).str.lower().eq("true") & df["landing_success"].notna()].copy()
labelled["Class"] = labelled["landing_success"].astype(int)
print("Launches:", len(df), "| labelled landing attempts:", len(labelled))
display(df.head())
display(labelled.groupby("launch_site")["Class"].agg(["count","mean"]).sort_values("mean", ascending=False))

In [ ]:
# 1. Flight number vs landing outcome
if not labelled.empty:
    plt.figure(figsize=(9,5))
    plt.scatter(labelled["flight_number"], labelled["Class"], alpha=.75)
    plt.xlabel("Flight number"); plt.ylabel("Landing success (1=yes, 0=no)")
    plt.title("Flight Number vs First-Stage Landing Outcome"); plt.yticks([0,1])
    plt.tight_layout(); plt.savefig(OUT/"flight_number_vs_outcome.png", dpi=160); plt.show()

In [ ]:
# 2. Payload mass vs landing outcome
plotdata = labelled.dropna(subset=["payload_mass_kg"])
if not plotdata.empty:
    plt.figure(figsize=(9,5))
    for cls, group in plotdata.groupby("Class"):
        plt.scatter(group["payload_mass_kg"], group["Class"], label=f"Class {cls}", alpha=.7)
    plt.xlabel("Payload mass (kg)"); plt.ylabel("Landing success (1=yes, 0=no)")
    plt.title("Payload Mass vs Landing Outcome"); plt.yticks([0,1]); plt.legend()
    plt.tight_layout(); plt.savefig(OUT/"payload_vs_outcome.png", dpi=160); plt.show()

In [ ]:
# 3. Launch-site counts and success rates
if not labelled.empty:
    stats = labelled.groupby("launch_site")["Class"].agg(["count","mean"]).sort_values("count", ascending=False)
    ax = stats["count"].plot(kind="bar", figsize=(10,5), title="Labelled Landing Attempts by Launch Site")
    ax.set_xlabel("Launch site"); ax.set_ylabel("Number of labelled attempts")
    plt.xticks(rotation=35, ha="right"); plt.tight_layout()
    plt.savefig(OUT/"launch_site_counts.png", dpi=160); plt.show()
    ax = (100*stats["mean"]).plot(kind="bar", figsize=(10,5), title="Landing Success Rate by Launch Site")
    ax.set_xlabel("Launch site"); ax.set_ylabel("Success rate (%)")
    plt.xticks(rotation=35, ha="right"); plt.tight_layout()
    plt.savefig(OUT/"launch_site_success_rate.png", dpi=160); plt.show()

In [ ]:
# 4. Yearly success trend
if not labelled.empty:
    yearly = labelled.groupby("year")["Class"].agg(["count","mean"]).dropna()
    yearly["success_rate_pct"] = yearly["mean"]*100
    display(yearly)
    ax = yearly["success_rate_pct"].plot(marker="o", figsize=(10,5), title="Yearly First-Stage Landing Success Rate")
    ax.set_xlabel("Year"); ax.set_ylabel("Success rate (%)")
    plt.tight_layout(); plt.savefig(OUT/"yearly_success_trend.png", dpi=160); plt.show()

In [ ]:
# 5. Orbit category (may contain multiple orbit names in a combined payload)
if not labelled.empty and "orbit" in labelled:
    orbit = labelled.assign(orbit=labelled["orbit"].fillna("Unknown")).groupby("orbit")["Class"].agg(["count","mean"]).sort_values("count", ascending=False).head(15)
    display(orbit)
    ax = (100*orbit["mean"]).plot(kind="bar", figsize=(11,5), title="Landing Success Rate by Orbit Label (Top 15 by Count)")
    ax.set_xlabel("Orbit label"); ax.set_ylabel("Success rate (%)")
    plt.xticks(rotation=40, ha="right"); plt.tight_layout()
    plt.savefig(OUT/"orbit_success_rate.png", dpi=160); plt.show()

## Interpretation prompts
After executing, write a short evidence-based observation below each chart. Mention sample sizes and avoid causal claims. Because this notebook uses live API data, the chart patterns may differ from the course's historical snapshot.